For each telugu article we find 5 english articles that are similar to it (cosine similarity) and we copy their interactions to this article 
after that, we bring telugu and english articles to a combined space.

In [4]:
import torch
import pickle
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import glob
import os
import gc


In [8]:
telugu_embeddings = torch.load("telugu_article_embeddings.pt")  # shape: (num_telugu, dim)
english_embeddings = torch.load("english_article_embeddings.pt")  # shape: (num_english, dim)


/var/folders/rw/b92f531j0kg9tv8t62n84v9c0000gn/T/ipykernel_67339/827554766.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  telugu_embeddings = torch.load("telugu_article

In [10]:
with open("english_news_ids.pkl", "rb") as f:
    english_ids = pickle.load(f)

with open("telugu_news_ids.pkl", "rb") as f:
    telugu_ids = pickle.load(f)

In [12]:
#loading click behaviours

mind_beh = pd.read_csv(
    '/Users/harshadayiniakula/Desktop/RS/MINDsmall_train/behaviors.tsv',
    sep='\t',
    header=None,
    names=['impression_id', 'user_id', 'timestamp', 'history', 'impressions'],
    dtype=str
)
mind_beh['history'] = mind_beh['history'].fillna('').apply(lambda x: x.strip().split())


In [13]:
# Create article_id → list of users who clicked
eng_article_to_users = {}

for uid, history in zip(mind_beh['user_id'], mind_beh['history']):
    for aid in history:
        if aid not in eng_article_to_users:
            eng_article_to_users[aid] = []
        eng_article_to_users[aid].append(uid)


In [14]:
english_matrix = torch.stack([english_embeddings[english_ids.index(aid)] for aid in english_ids]).numpy()
telugu_matrix = telugu_embeddings.numpy()


In [19]:

k = 5
batch_size = 500
num_telugu = len(telugu_matrix)

for start in tqdm(range(0, num_telugu, batch_size), desc="Processing Telugu batches"):
    end = min(start + batch_size, num_telugu)
    inherited_rows = []

    for i in range(start, end):
        telugu_id = telugu_ids[i]
        tel_vec = telugu_matrix[i]

        sims = cosine_similarity([tel_vec], english_matrix)[0]
        top_k_idx = np.argsort(sims)[-k:][::-1]

        for idx in top_k_idx:
            eng_id = english_ids[idx]
            sim_score = sims[idx]

            if eng_id not in eng_article_to_users:
                continue

            for user_id in eng_article_to_users[eng_id]:
                inherited_rows.append({
                    'user_id': user_id,
                    'article_id': telugu_id,
                    'clicked': 1,
                    'inherited_from': eng_id,
                    'similarity_score': sim_score
                })

    # Save this batch immediately
    batch_df = pd.DataFrame(inherited_rows)
    batch_df.to_csv(f"inherited_telugu_interactions_{start}_{end}.csv", index=False)
    print(f"Saved {len(batch_df)} interactions for batch {start}–{end}")

    # Clear memory
    del inherited_rows, batch_df
    gc.collect()


Processing Telugu batches:   0%|                        | 0/193 [00:06<?, ?it/s]


KeyboardInterrupt: 

In [24]:
# Change to your folder if needed
folder = "/Users/harshadayiniakula/Desktop/RS"

# Grab all matching batch CSV files
all_files = sorted(glob.glob(os.path.join(folder, "inherited_telugu_interactions_*.csv")))

# Load and  concatenate
df_list = [pd.read_csv(file) for file in all_files]
final_df = pd.concat(df_list, ignore_index=True)

# Save the full merged file
final_df.to_csv(os.path.join(folder, "inherited_telugu_interactions.csv"), index=False)
print(f"Merged {len(all_files)} batch files into inherited_telugu_interactions.csv with {len(final_df)} rows.")


Merged 193 batch files into inherited_telugu_interactions.csv with 258807200 rows.


### Combining english and inherited telugu interactions into a single space

In [4]:
mind_beh_original = pd.read_csv('/Users/harshadayiniakula/Desktop/RS/MINDsmall_train/behaviors.tsv')

In [12]:
print(mind_beh_original.head(5))

  1\tU13740\t11/11/2019 9:05:58 AM\tN55189 N42782 N34694 N45794 N18445 N63302 N10414 N19347 N31801\tN55689-1 N35729-0
0  2\tU91836\t11/12/2019 6:11:30 PM\tN31739 N6072...                                                                 
1  3\tU73700\t11/14/2019 7:01:48 AM\tN10732 N2579...                                                                 
2  4\tU34670\t11/11/2019 5:28:05 AM\tN45729 N2203...                                                                 
3  5\tU8125\t11/12/2019 4:11:21 PM\tN10078 N56514...                                                                 
4  6\tU19739\t11/11/2019 6:52:13 PM\tN39074 N1434...                                                                 


In [14]:
mind_beh = pd.read_csv(
    '/Users/harshadayiniakula/Desktop/RS/MINDsmall_train/behaviors.tsv',
    sep='\t',
    header=None,
    names=['impression_id', 'user_id', 'timestamp', 'history', 'impressions'],
    dtype=str
)

# Fill NA with empty list and split history into article IDs
mind_beh['history'] = mind_beh['history'].fillna('').apply(lambda x: x.strip().split())

# Build the interaction dataframe from history
interaction_rows = []

for uid, clicks in zip(mind_beh['user_id'], mind_beh['history']):
    for aid in clicks:
        interaction_rows.append({
            'user_id': uid,
            'article_id': aid,
            'clicked': 1
        })

english_df = pd.DataFrame(interaction_rows)

# Save for future use
english_df.to_csv("/Users/harshadayiniakula/Desktop/RS/english_interactions.csv", index=False)
print(f"Saved {len(english_df)} English interactions.")


Saved 5107639 English interactions.


In [7]:
import pandas as pd

# Load inherited interactions (Telugu)
telugu_df = pd.read_csv("/Users/harshadayiniakula/Desktop/RS/inherited_telugu_interactions.csv")

# Load original English interactions (real clicks)
english_df = pd.read_csv("/Users/harshadayiniakula/Desktop/RS/english_interactions.csv")  # update if path is different

# Standardize columns: user_id, article_id, clicked
english_df = english_df[['user_id', 'article_id', 'clicked']]

# Drop extra columns in Telugu if needed
telugu_df = telugu_df[['user_id', 'article_id', 'clicked']]

# Combine both
combined_df = pd.concat([english_df, telugu_df], ignore_index=True)

# Optional: Save to disk
combined_df.to_csv("/Users/harshadayiniakula/Desktop/RS/combined_interactions.csv", index=False)
print(f"Final interaction matrix contains {len(combined_df)} rows.")


Final interaction matrix contains 263914839 rows.


### A way to filter telugu articles and inherit interactions (threshold = 0.70)
I felt like the telugu dataset was unnecessarily large. 
If a Telugu article isn’t at least, say, 70% “similar” (in meaning) to any English piece, it’s probably off-topic or too niche—and borrowing clicks for it would be guessing.
By dropping these low-similarity articles, we are drastically cutting down on noise and on the number of rows we have to process.
All the inherited interactions now come from articles that truly “match” in content, so your model learns real user preferences instead of fabricating connections.


In [22]:
theta = 0.70   

# 1) Compute max similarity per Telugu article
max_sims = []
for tel_vec in tqdm(telugu_matrix, desc="Computing max similarity"):
    sims = cosine_similarity([tel_vec], english_matrix)[0]
    max_sims.append(np.max(sims))
max_sims = np.array(max_sims)

# 2) Build mask of articles to keep 
keep_mask = max_sims >= theta
kept_indices = np.where(keep_mask)[0]

print(f"Threshold θ = {theta}")
print(f"  Kept {len(kept_indices)} out of {len(telugu_ids)} Telugu articles")

# 3) Filter the matrices and ID list
filtered_telugu_matrix = telugu_matrix[keep_mask]
filtered_telugu_ids    = [telugu_ids[i] for i in kept_indices]

# 4) Save filtered artifacts for next steps
torch.save(torch.from_numpy(filtered_telugu_matrix), "filtered_telugu_embeddings.pt")
with open("filtered_telugu_ids.pkl", "wb") as f:
    pickle.dump(filtered_telugu_ids, f)

print("Saved:")
print(" • filtered_telugu_embeddings.pt")
print(" • filtered_telugu_ids.pkl")

Computing max similarity: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 96370/96370 [49:33<00:00, 32.41it/s]

Threshold θ = 0.7
  Kept 13 out of 96370 Telugu articles
Saved:
 • filtered_telugu_embeddings.pt
 • filtered_telugu_ids.pkl


#### Threshold change to 0.60
keeping the threshold at 0.70 resulted in only 13 articles i.e., only 13 articles have cosine similarity > 0.70 to any other article in the english dataset. To have more articles in the dataset, I am experimenting with threshold 0.60. 

In [21]:
theta = 0.60   # similarity threshold; adjust between 0.65–0.80 as you like

# ——— 1) Compute max similarity per Telugu article ———
max_sims = []
for tel_vec in tqdm(telugu_matrix, desc="Computing max similarity"):
    sims = cosine_similarity([tel_vec], english_matrix)[0]
    max_sims.append(np.max(sims))
max_sims = np.array(max_sims)

# ——— 2) Build mask of articles to keep ———
keep_mask = max_sims >= theta
kept_indices = np.where(keep_mask)[0]

print(f"Threshold θ = {theta}")
print(f"  Kept {len(kept_indices)} out of {len(telugu_ids)} Telugu articles")

# ——— 3) Filter the matrices and ID list ———
filtered_telugu_matrix = telugu_matrix[keep_mask]
filtered_telugu_ids    = [telugu_ids[i] for i in kept_indices]

# ——— 4) Save filtered artifacts for next steps ———
torch.save(torch.from_numpy(filtered_telugu_matrix), "filtered_0.6_telugu_embeddings.pt")
with open("filtered_0.6_telugu_ids.pkl", "wb") as f:
    pickle.dump(filtered_telugu_ids, f)

print("Saved:")
print(" • filtered_0.6_telugu_embeddings.pt")
print(" • filtered_0.6_telugu_ids.pkl")

Computing max similarity: 100%|███████████| 96370/96370 [50:29<00:00, 31.81it/s]

Threshold θ = 0.6
  Kept 4316 out of 96370 Telugu articles
Saved:
 • filtered_0.6_telugu_embeddings.pt
 • filtered_0.6_telugu_ids.pkl


In [23]:
# 1) Load filtered Telugu embeddings & IDs
telugu_embeddings = torch.load("filtered_0.6_telugu_embeddings.pt")    # (4316, dim)
with open("filtered_0.6_telugu_ids.pkl", "rb") as f:
    telugu_ids = pickle.load(f)                                        # list of 4316 IDs

# 2) Load full English embeddings & IDs
english_embeddings = torch.load("english_article_embeddings.pt")      # (N_eng, dim)
with open("english_news_ids.pkl", "rb") as f:
    english_ids = pickle.load(f)                                      

eng_matrix = english_embeddings.numpy()
tel_matrix = telugu_embeddings.numpy()

# 3) Build English article → clicked-user list
mind_beh = pd.read_csv(
    '/Users/harshadayiniakula/Desktop/RS/MINDsmall_train/behaviors.tsv',
    sep='\t', header=None,
    names=['impression_id','user_id','timestamp','history','impressions'],
    dtype=str
)
mind_beh['history'] = mind_beh['history'].fillna('').str.split()
eng_to_users = {}
for uid, hist in zip(mind_beh['user_id'], mind_beh['history']):
    for aid in hist:
        eng_to_users.setdefault(aid, []).append(uid)

# 4) Inherit interactions
k = 5
inherited = []
for tel_vec, tel_id in tqdm(zip(tel_matrix, telugu_ids), total=len(telugu_ids), desc="Inheriting clicks"):
    sims = cosine_similarity([tel_vec], eng_matrix)[0]
    top_k = sims.argsort()[-k:][::-1]
    for idx in top_k:
        eng_id = english_ids[idx]
        if eng_id not in eng_to_users: 
            continue
        sim_score = float(sims[idx])
        for user in eng_to_users[eng_id]:
            inherited.append({
                'user_id': user,
                'article_id': tel_id,
                'clicked': 1,
                'inherited_from': eng_id,
                'similarity_score': sim_score
            })

# 5) Save the result
inherited_df = pd.DataFrame(inherited)
inherited_df.to_csv("inherited_telugu_interactions_0.6.csv", index=False)
print(f"Saved {len(inherited_df):,} inherited interactions for 4,316 Telugu articles.")


/var/folders/rw/b92f531j0kg9tv8t62n84v9c0000gn/T/ipykernel_67339/2136906088.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  telugu_embeddings = torch.load("filtered_0.6_

Saved 12,336,253 inherited interactions for 4,316 Telugu articles.


### What interactions to inherit ? 
I feel like inheriting all interactions of k closest neighbours would increase noise and create synthetic data not backed by ground truth. 
I am exploring strategies to cap this and only inherit the most meaningful interactions

also, capping number of users per neighbor article to 100. Ex: even if an english article has around 500 interactions, only 100 would be inherited. 

In [27]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import pandas as pd
import pickle, torch, random

# ---------- 1.  LOAD EMBEDDINGS & USER MAP  ----------
telugu_embeddings = torch.load("filtered_0.6_telugu_embeddings.pt").numpy()
with open("filtered_0.6_telugu_ids.pkl", "rb") as f:
    telugu_ids = pickle.load(f)

english_embeddings = torch.load("english_article_embeddings.pt").numpy()
with open("english_news_ids.pkl", "rb") as f:
    english_ids = pickle.load(f)

# Build English‑article → list‑of‑users map (already done once; reload if needed)
mind_beh = pd.read_csv(
    '/Users/harshadayiniakula/Desktop/RS/MINDsmall_train/behaviors.tsv',
    sep='\t', header=None,
    names=['impression_id','user_id','timestamp','history','impressions'],
    dtype=str
)
mind_beh['history'] = mind_beh['history'].fillna('').str.split()
eng_to_users = {}
for uid, hist in zip(mind_beh['user_id'], mind_beh['history']):
    for aid in hist:
        eng_to_users.setdefault(aid, []).append(uid)

# Pre‑compute: number‑of‑users for every English article
eng_click_counts = {aid: len(u) for aid, u in eng_to_users.items()}

# ---------- 2.  PARAMETERS TO TEST  ----------
thresholds          = [0.60, 0.65, 0.70, 0.75, 0.80]   # neighbour similarity cut‑offs
k_neighbours        = 5                                # top‑k neighbours you’ll consider
max_users_per_neigh = 100                              # cap users copied from each neighbour

# ---------- 3.  SWEEP & COLLECT STATS  ----------
rows = []
for theta_n in thresholds:
    telugu_with_clicks = 0
    total_synthetic    = 0
    
    for tel_vec in tqdm(telugu_embeddings, leave=False):
        sims = cosine_similarity([tel_vec], english_embeddings)[0]
        top_k_idx = sims.argsort()[-k_neighbours:][::-1]   # indices of k best english neighbours
        
        this_article_rows = 0
        for idx in top_k_idx:
            sim = sims[idx]
            if sim < theta_n:             # neighbour too weak
                continue
            eng_id = english_ids[idx]
            clicks_here = eng_click_counts.get(eng_id, 0)
            
            # apply per‑neighbour cap
            clicks_here = min(clicks_here, max_users_per_neigh)
            this_article_rows += clicks_here
        
        if this_article_rows > 0:
            telugu_with_clicks += 1
            total_synthetic   += this_article_rows
    
    rows.append({
        "θₙ": theta_n,
        "Telugu_articles_with_clicks": telugu_with_clicks,
        "Synthetic_rows": total_synthetic,
        "Avg_rows_per_Telugu": round(total_synthetic / telugu_with_clicks, 2) if telugu_with_clicks else 0
    })

# ---------- 4.  DISPLAY SUMMARY  ----------
summary_df = pd.DataFrame(rows)
summary_df



/var/folders/rw/b92f531j0kg9tv8t62n84v9c0000gn/T/ipykernel_67339/1467841954.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  telugu_embeddings = torch.load("filtered_0.6_

,θₙ,Telugu_articles_with_clicks,Synthetic_rows,Avg_rows_per_Telugu
0,0.60,4015,377001,93.90
1,0.65,130,6300,48.46
2,0.70,13,33,2.54
3,0.75,0,0,0.00
4,0.80,0,0,0.00


Decided to keep the threshold to 0.60 (neighbours must be atleast 0.60 similar along with being top 3 closest neighbours  to the telugu article for inheriting their clicks 

Saved 334,499 synthetic clicks.
